# Quick EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from io import StringIO

data_dir = Path("data")

def read_bfs_csv(path):
    lines = path.read_text(encoding="latin1").splitlines()
    text = "\n".join(line[1:-1].replace('""', '"') for line in lines)
    return pd.read_csv(StringIO(text))

def overview(df, key_columns):
    return pd.DataFrame({
        "Rows": [df.shape[0]],
        "Columns": [df.shape[1]],
        "Missing key columns (%)": [round(df[key_columns].isna().mean().mean() * 100, 2)]
    })

antrieb = read_bfs_csv(data_dir / "Antriebsarten nach Gemeinden.csv")
einwohner = read_bfs_csv(data_dir / "Einwohner nach Gemeinden.csv")
geo = pd.read_excel(data_dir / "Geographische_Kennzahlen.xlsx", sheet_name="Daten")

## 1. Antriebsarten nach Gemeinden

In [ ]:
overview(antrieb, ["Gemeinde", "Fahrzeuggruppe", "Treibstoff", "2024"] )

In [ ]:
antrieb[["2010", "2024"]].agg(["mean", "median", "min", "max"]).round(1)

In [ ]:
(antrieb[antrieb["Fahrzeuggruppe"] == "Personenwagen"]
 .groupby("Treibstoff")[["2010", "2024"]]
 .sum()
 .sort_values("2024", ascending=False)
 .head())

## 2. Einwohner nach Gemeinden

In [ ]:
overview(einwohner, ["Jahr", "Kanton (-) / Bezirk (>>) / Gemeinde (......)", "Staatsangehörigkeit - Total"] )

In [ ]:
gemeinden = einwohner[einwohner.iloc[:, 1].str.startswith("......")]
gemeinden_2024 = gemeinden[gemeinden["Jahr"] == 2024]
gemeinden_2024["Staatsangehörigkeit - Total"].agg(["mean", "median", "min", "max"]).round(1).to_frame("Einwohner 2024")

In [ ]:
schweiz = (einwohner[einwohner.iloc[:, 1] == "Schweiz"]
           .set_index("Jahr")["Staatsangehörigkeit - Total"])
pd.DataFrame({
    "Einwohner": schweiz.loc[[2010, 2024]],
    "Veränderung seit 2010 (%)": [0, round((schweiz.loc[2024] / schweiz.loc[2010] - 1) * 100, 1)]
})

## 3. Geographische Kennzahlen

In [ ]:
overview(geo, ["GDENR", "GDENAME", "AREA_HA", "KTNAME"] )

In [ ]:
geo["AREA_HA"].agg(["mean", "median", "min", "max"]).round(1).to_frame("Fläche (ha)")

In [ ]:
geo.nlargest(5, "AREA_HA")[["GDENAME", "KTNAME", "AREA_HA"]]

## Kleiner Vergleich

In [ ]:
pd.DataFrame({
    "Rows": [len(antrieb), len(einwohner), len(geo)],
    "Columns": [antrieb.shape[1], einwohner.shape[1], geo.shape[1]],
    "Missing (%)": [antrieb.isna().mean().mean() * 100, einwohner.isna().mean().mean() * 100, geo.isna().mean().mean() * 100]
}, index=["Antriebsarten", "Einwohner", "Geographie"]).round(2)

In [ ]:
antrieb["GDENR"] = antrieb["Gemeinde"].str.extract(r"^(\d+)").astype(int)
einwohner["GDENR"] = einwohner.iloc[:, 1].str.extract(r"^\.{6}(\d+)")

population = (einwohner[einwohner["Jahr"] == 2024]
              .dropna(subset=["GDENR"])
              .assign(GDENR=lambda x: x["GDENR"].astype(int))
              .groupby("GDENR", as_index=False)["Staatsangehörigkeit - Total"]
              .sum()
              .rename(columns={"Staatsangehörigkeit - Total": "Einwohner_2024"}))

personenwagen = (antrieb[antrieb["Fahrzeuggruppe"] == "Personenwagen"]
                 .groupby("GDENR", as_index=False)["2024"]
                 .sum()
                 .rename(columns={"2024": "Personenwagen_2024"}))

vergleich = population.merge(personenwagen, on="GDENR")
korrelation = vergleich["Einwohner_2024"].corr(vergleich["Personenwagen_2024"])
pd.DataFrame({"Korrelation Einwohner / Personenwagen 2024": [korrelation]}).round(3)

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(vergleich["Einwohner_2024"], vergleich["Personenwagen_2024"], alpha=0.35, s=15)
plt.xlabel("Einwohner 2024")
plt.ylabel("Personenwagen 2024")
plt.title(f"Korrelation: {korrelation:.3f}")
plt.tight_layout()
plt.show()